In [ ]:
import os
import numpy as np
import xarray as xr
import zarr

import dask
import matplotlib.pyplot as plt

from dask.diagnostics import ProgressBar

In [ ]:
ddss = xr.open_zarr('/mnt/tier1/project/p200177/DE_371_bis/meps-2p5km-2020-2025-1h-v2_subdomain_subvars_rechu.zarr', zarr_format=2)

In [ ]:
ddss.attrs['variables']

In [ ]:
idx = ddss.attrs['variables'].index("2t")


In [ ]:
(((31956-6)//6)-2)%3

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# 1. Extraction (Using your indices)
# Note: we need 82 points to get 144 differences
#raw_subset = ddss['data'].isel(variable=idx, ensemble=0, time=slice(31956-6, 31956-6+145)).compute()
raw_subset = ddss['data'].isel(variable=idx, ensemble=0, time=slice(0, 145)).compute()

images_2d = raw_subset.values.reshape(145, 256, 256)

v_min, v_max = -5, 5

# 2. Setup the grid (12 rows, 7 columns as per your request)
fig, axes = plt.subplots(23, 7, figsize=(25, 40))
axes_flat = axes.flatten()

# Define the cycle colors (e.g., Cycle 1: Red, Cycle 2: Green, Cycle 3: Blue)
# This will repeat every 18 hours (3 blocks of 6)
cycle_colors = ['#00FF00', '#00FF00', '#FF4B4B'] 

for i in range(144):
    ax = axes_flat[i]
    
    # Calculate the difference: Hour(i+1) - Hour(i)
    diff_img = images_2d[i+1] - images_2d[i]
    im = ax.imshow(diff_img, cmap='bwr', vmin=v_min, vmax=v_max, origin='lower')
    
    # Determine which 6-hour block we are in (0, 1, or 2)
    # i // 6 gives the block number, % 3 makes it repeat the 3 colors
    color_idx = (i // 6) % 3
    current_color = cycle_colors[color_idx]
    
    # --- ADDING THE COLORED BORDER ---
    # We turn the axis 'on' but hide ticks to show the spine (border)
    ax.axis('on')
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Increase border thickness and set color
    for spine in ax.spines.values():
        spine.set_edgecolor(current_color)
        spine.set_linewidth(3 if (i+1)%6 == 0 else 1.5) # Thicker border on boundary hour
        
    # --- LABELING ---
    if (i+1) % 6 == 0:
        ax.set_title(f"H{i} (Bound)", fontsize=12, fontweight='bold', color=current_color)
    else:
        ax.set_title(f"H{i}", fontsize=9, color='gray')

# Hide unused axes if any
for j in range(144, len(axes_flat)):
    axes_flat[j].axis('off')

# Colorbar and Titles
cbar = fig.colorbar(im, ax=axes, location='right', shrink=0.1)
cbar.set_label('$\Delta$ Temperature (K)', rotation=270, labelpad=15)
plt.suptitle("Hourly Temperature Differences with 6-Hour Cycle Grouping", fontsize=22, y=1.02)


plt.savefig('differences.png',             dpi=300)

In [ ]:
print(ddss)

In [ ]:
time_range=range(1025)
time_rangem1=range(1024)

In [ ]:
ddss['dates'].min().compute()

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt


# 1. Calculate the difference between successive time steps.
# Using .diff('time') calculates (time[t] - time[t-1]), reducing the time dimension size by 1 (1440 -> 1439).
data = ddss['data'].isel(variable=idx, ensemble=0, time=time_range)
delta_2t = ddss['data'].isel(variable=idx, ensemble=0, time=time_range).diff(dim='time')
 

In [ ]:
boundary_diffs = data.isel(time=slice(6, None, 6)).values - \
                  data.isel(time=slice(5, None, 6)).values

In [ ]:
boundary_diffs

In [ ]:

# 2. Calculate the Mean Squared Difference
# Square the differences and take the mean across all spatial/variable/ensemble dimensions
mean_squared_diff = (boundary_diffs ** 2).mean(1)
mean_squared_diff.shape

In [ ]:

# 5. Plot the results
plt.figure(figsize=(14, 6))
plt.plot( mean_squared_diff, color='#1f77b4', linewidth=1.5, label='Mean Squared Successive Difference')

plt.title('Mean Squared Difference Between Successive Time Steps (1h intervals)', fontsize=14)
plt.xlabel('Time', fontsize=12)
plt.ylabel('Mean Squared Difference', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:

# 3. Trigger Dask Computation
# Best Practice: Explicitly compute the 1D array into memory before passing to plotting libraries.
# This prevents matplotlib from implicitly triggering unpredictable and unoptimized Dask evaluations.
msd_values = mean_squared_diff.compute()
msd_values.shape

In [ ]:

# 4. Extract matching dates for the X-axis
# Since diff() drops the first time index, we align our dates by slicing from index 1 to the end.
plot_dates = ddss['dates'].isel(time=time_rangem1).compute()
plot_dates.shape

In [ ]:

# 5. Plot the results
plt.figure(figsize=(14, 6))
plt.plot(time_rangem1, msd_values, color='#1f77b4', linewidth=1.5, label='Mean Squared Successive Difference')

plt.title('Mean Squared Difference Between Successive Time Steps (1h intervals)', fontsize=14)
plt.xlabel('Time', fontsize=12)
plt.ylabel('Mean Squared Difference', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# 1. Ensure your MSD data is a 1D numpy array of size 1024
# signal = msd_values.values[:1024] 

# 2. Calculate the Real FFT components
# This returns 513 complex numbers (n/2 + 1)
fft_complex = np.fft.rfft(msd_values)

# 3. Get the Magnitude (The "Components")
# This gives you the strength of each frequency
amplitudes = np.abs(fft_complex)

# 4. Get the corresponding frequencies
# d=1.0 assumes your sampling is 1 unit (e.g., 1 hour)
freqs = np.fft.rfftfreq(1024, d=1.0)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Prepare your 1024-point numpy array
# data_1d = msd_values.values[:1024]

# 2. Compute the Real FFT
fft_values = np.fft.rfft(msd_values)
amplitudes = np.abs(fft_values)

# 3. Compute frequencies
freqs = np.fft.rfftfreq(1024, d=1.0)

# 4. Convert Frequency to Period (Hours)
# We skip the first element (index 0) to avoid divide by zero
periods = 1 / freqs[1:]
amplitudes_plot = amplitudes[1:]

# 5. Plot
plt.figure(figsize=(12, 6))

# Use a log scale for the X-axis (Periods) because weather 
# cycles often span orders of magnitude (e.g., 2h to 500h).
plt.semilogx(periods, amplitudes_plot, color='tab:red', linewidth=1.5)

# Add vertical lines for common meteorological cycles
plt.axvline(24, color='k', linestyle='--', alpha=0.5, label='24h (Diurnal)')
plt.axvline(12, color='blue', linestyle=':', alpha=0.5, label='12h (Semi-diurnal)')

plt.title("FFT Spectrum: Amplitude vs. Period", fontsize=14)
plt.xlabel("Period [Hours per Cycle]", fontsize=12)
plt.ylabel("Amplitude", fontsize=12)

# Set x-ticks to common hourly values for better readability
plt.xticks([1, 2, 6, 12, 24, 48, 168, 1024], 
           ['1h', '2h', '6h', '12h', '24h', '48h', '1 week', 'End'])

plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()